# H1 — Thinking-Budget Study

**Models:** `LiquidAI/LFM2.5-2.6B` · `Qwen/Qwen3-0.6B` · `deepseek-ai/DeepSeek-R1-Distill-Llama-8B`
**Platform:** Kaggle 2×T4 (30 GPU-hours/week, no billing setup), vLLM with a thinking-budget logits processor.

One self-contained notebook runs the complete experiment for any of the three models. The model is selected in a single place (§4); everything else — benchmarks, graders, evaluator, sweep, checkpointing — is shared, so results are directly comparable across models.

## Purpose

This is a **calibration** study, not only an "accuracy rises with more thinking" study:

> Does the model spend reasoning tokens where they actually buy accuracy, or does it **overthink** (burn budget on problems already solved at a small budget) and **underthink** (run out of budget before it could have answered correctly)?

Two sub-questions follow:

1. **Quantity** — at which budget does each benchmark's accuracy curve flatten (the *knee*), and does the knee move with difficulty the way it should?
2. **Quality** — does a longer trace reason *better*, or does it loop and thrash?

## Research design

**Independent variable: a hard cap on thinking tokens.** The cap is enforced at decoding time, not requested in the prompt. A decoding-time cap changes only the compute; prompt-level length control would confound reasoning ability with instruction following. When the cap is reached, the serving layer forces the model to leave its reasoning segment and answer.

```
9 benchmarks × 100 questions × 7 compute levels × 5 samples ≈ 29,050 calls per model
(AIME 2025 has 30 questions: 8×100×7×5 + 30×7×5)
COMPUTE_LEVELS = [0, 256, 512, 1024, 2048, 4096, 8192]
```

- **Log-spaced budgets.** Accuracy versus compute is roughly logarithmic; linear spacing would waste most points in the flat region. 256 approximates "no room to think" (a proxy for raw difficulty); 8192 is the practical ceiling for the context length used. Budget 0 forces an immediate switch to the answer and gives the no-reasoning baseline.
- **Five samples per (question, budget) at `temperature = 0.7`.** One sample is a coin flip on a hard item; five give a per-question accuracy *and* a stability measure. At temperature 0 the five samples would be identical and variance could not be measured.
- **Nine benchmarks in three difficulty tiers.** The hypothesis concerns the *relationship* between difficulty and the optimal budget, which needs at least an ordinal difficulty axis.

## Contents

| § | Section |
|---|---|
| 1 | Environment |
| 2 | Serving layer |
| 3 | Models and thinking tokens |
| 4 | Launch and pre-flight check |
| 5 | Benchmark suite |
| 6 | Data collection |
| 7 | Pilot |
| 8 | Full sweep |
| 9 | Evaluation methodology |
| 10 | Limitations |
| 11 | What's Next — textual analysis of `thinking_text` |

## Running it

1. **Kaggle → Code → New Notebook**, then *File → Import/Upload Notebook*.
2. **Settings → Accelerator:** `GPU T4 ×2`. **Internet ON.**
3. **(GPQA only)** Settings → **Secrets** → add `HF_TOKEN`, and accept the terms at <https://huggingface.co/datasets/Idavidrein/gpqa> with that account.
4. **Choose the model:** set `MODEL_KEY` in the launch cell (§4) to `"lfm2.5-2.6b"`, `"qwen3-0.6b"` or `"deepseek-r1-distill-llama-8b"`.
5. **Run all.** Installation takes ≈ 5 minutes the first time; servers 2–5 minutes (longer for the 8B, whose weights are ≈ 16 GB); the pilot under a minute for the small models; the full sweep is a multi-hour, decode-bound job and is fully resumable.
6. **Results:** Output tab → *Download All* — `h1_raw_results.csv`, `h1_pilot_results.csv`, `sample2.csv`. Progress is also appended to checkpoint files during the run (`h2_pilot_results.csv`, `h2_raw_results.csv`).

Each model is run as its own Kaggle session and its exports are kept separate. Every row carries a `model` column, and a checkpoint belonging to a different model is rejected on resume, so results from different models can never mix silently. If a session drops, **Run all** again: the sweep resumes from its checkpoint.

## Inference design at a glance

The sweep is ≈ 29,000 generations of up to 8K thinking tokens each, so throughput and robustness are part of the method. Each design decision and the concern it answers:

| Concern | Design |
|---|---|
| A fixed number of client workers leaves the GPU's KV cache idle | **KV-aware admission control**: each request is charged an expected KV footprint and is admitted to a server only while its estimated load stays under that server's KV capacity. |
| Long requests (budget 4096/8192, ≈ 5 tok/s each) can end a run on a lone straggler | **Shuffled scheduling** mixes budgets so in-flight requests are at different progress points and the average KV estimate holds. |
| Default HTTP client timeouts (600 s) plus SDK retries kill and re-send long generations from scratch, which looks like a stall | **`timeout = 7200 s`, `max_retries = 0`**, with an explicit, bounded retry only for connection errors, timeouts, 5xx and rate limits. |
| Rewriting a CSV that contains hundreds of MB of thinking text is slow and holds the whole result list in RAM | **Append-only checkpoint** every 50 rows or 60 s; only the key columns are loaded on resume. |
| Recomputing statistics over all rows on every completion starves the HTTP threads | **O(1) running counters** for accuracy, marker hit-rate and throughput. |
| Triton kernels compile in the middle of the run, causing latency spikes | **Warm-up requests** of varied length right after the servers start. |
| A stale engine from a previous run blocks the new server's handshake | **Wait for GPU memory to be released** after killing old processes; **staggered but overlapping** launch of the servers. |
| Silent stalls | **60-second heartbeat** with in-flight requests, KV load, `/health`, tokens/s and ETA; clean abort after 20 consecutive failures with progress saved. |

## 1. Environment

Three things are installed, each for a specific reason:

| Component | Role |
|---|---|
| **`vllm-thinking-budget` fork** | Stock vLLM has no per-request thinking cap. The fork adds a logits processor that watches generated tokens and, at `budget − 1`, forces a newline and the closing boundary token, so the model *must* stop thinking and answer. It runs against a stock `vllm` wheel (≥ 0.17) through `PYTHONPATH`; no source build is needed. |
| **`torchaudio` removal** | The Kaggle base image bundles a build that conflicts with the torch version the vLLM wheel pins. It is unused here. |
| **Sparse clone of `gorilla`** | BFCL data only (the `bfcl_eval/data` subtree). A full clone is large and slow. |
| **`kneed`** | Knee detection on accuracy-versus-budget curves (§9). |

`HF_TOKEN` is optional and only needed for the gated GPQA dataset.

In [1]:
# Thinking-budget vLLM fork (works with stock `vllm` >=0.17 via PYTHONPATH).
!git clone --depth 1 https://github.com/phishingupstream/vllm-thinking-budget.git

# Runtime deps. This pulls/upgrades torch to match the vLLM wheel.
!pip install -q -U vllm datasets openai "pandas<3" "numpy==2.3.5" matplotlib "kneed"

# Kaggle base image bundles torchaudio which conflicts with the pinned torch —
# unused here, so remove it.
!pip uninstall -y -q torchaudio

# BFCL v4 data: sparse clone of ONLY the bfcl_eval/data subtree (fast, small).
!git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/ShishirPatil/gorilla.git /kaggle/working/gorilla
!git -C /kaggle/working/gorilla sparse-checkout set \
    berkeley-function-call-leaderboard/bfcl_eval/data

# HF login for the gated GPQA dataset.
import os
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    print("[hf] HF_TOKEN loaded from Kaggle secrets")
except Exception as e:
    print(f"[hf] no HF_TOKEN secret: {e}")

import vllm, datasets, openai, pandas
print(f"vllm={vllm.__version__} datasets={datasets.__version__}")

# Sanity: the fork's processor file must be importable.
print("fork present:",
      os.path.isdir("/kaggle/working/vllm-thinking-budget"),
      os.path.exists("/kaggle/working/vllm-thinking-budget/thinking_budget_processor.py"))
print("gorilla BFCL data present:",
      os.path.exists("/kaggle/working/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v4_simple_python.json"))

fatal: destination path 'vllm-thinking-budget' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/gorilla' already exists and is not an empty directory.
[hf] HF_TOKEN loaded from Kaggle secrets
vllm=0.31.0 datasets=5.1.0
fork present: True True
gorilla BFCL data present: True


In [2]:
import numpy
numpy.__version__

'2.3.5'

## 2. Serving layer — vLLM servers, one layout per model

### Data parallelism for models that fit one GPU

A 2.6B model fits entirely on one 16 GB T4, and so does a 0.6B model. Splitting either with tensor parallelism (TP = 2) would add an all-reduce per layer across two GPUs: pure communication overhead for a memory problem that does not exist. The right way to use N GPUs is **data parallelism** — one independent vLLM instance per GPU (TP = 1 each), with the evaluator round-robining requests across `server.ports`. That gives ≈ 2× throughput instead of ≈ 1× with extra latency. With a single T4 the same code runs on one server.

### Tensor parallelism for the 8B model

DeepSeek-R1-Distill-Llama-8B needs ≈ 16 GB of fp16 weights, more than a T4 can hold (the T4 has no bf16 or FP8 support, so fp16 is used). It is therefore served as **one TP = 2 instance spanning both GPUs**. Roughly 10 GB remains for the KV cache across the pair, so only a handful of 8K-token traces run concurrently and all-reduce traffic crosses PCIe; throughput is the lowest of the three models and the grid takes correspondingly longer. The evaluator adapts automatically because its admission limits are derived from the KV-cache size the server reports.

| Model | Layout | Notes |
|---|---|---|
| LFM2.5-2.6B | 2 servers, one per T4, TP = 1 | ≈ 5.2 GB fp16 leaves a large KV cache. |
| Qwen3-0.6B | 2 servers, one per T4, TP = 1 | Tiny weights; the KV cache dominates. Native 32K context equals the configured maximum. |
| DeepSeek-R1-Distill-Llama-8B | **1 server, TP = 2 over both T4s** | Longer startup (weight download and load); lower concurrency. |

A 4-bit quantized checkpoint could restore data parallelism for the 8B, but it changes the model under study and must be recorded as a confound; fp16 with TP = 2 is the reference configuration.

### The budget-processor guard

Under vLLM's async scheduling, the token buffer the thinking-budget processor scans can contain `-1` placeholders for positions that are not committed yet. An unguarded processor treats `-1` as a real token id and miscounts how much thinking has happened, so the budget stops binding correctly. `apply_patch()` makes the scan stop at the first `-1`; it is idempotent (an already-patched file is detected and left alone). Each request carries its budget as `vllm_xargs = {"max_thinking_tokens": budget}`.

### Server configuration

| Setting | Value |
|---|---|
| `max-model-len` | 32768 (16384 if VRAM per GPU is below 12 GB) |
| `gpu-memory-utilization` | 0.95 (0.92 on low VRAM) |
| `max-num-seqs` | 265 (128 on low VRAM) |
| `tensor-parallel-size` | 1 for the small models, 2 for the 8B |
| `enable-prefix-caching` | true |
| `logits-processors` | `thinking_budget_processor:ThinkingBudgetLogitsProcessor` |

`enable-prefix-caching` matters more than usual: every question is sent once per budget and sample (35 times across the grid), so caching the prompt prefill makes the sweep decode-bound rather than prefill-bound. Context length and memory utilization step down automatically when VRAM per GPU is low, so the notebook degrades instead of failing with an out-of-memory error.

### Server lifecycle

1. Kill stale vLLM processes and **wait until GPU memory is actually released**.
2. Launch servers staggered but overlapping: the first finishes `torch.compile` before the next starts, so the next reuses the compile cache; everything after that overlaps.
3. Poll `/health` on every port; if a process exits, print the tail of its log and abort.
4. Read each server's **KV-cache size in tokens** from its log; admission control uses it.
5. **Warm up** every server with concurrent requests of varied prompt length so Triton kernels and CUDA-graph shapes compile before the sweep rather than as a latency spike in the middle of it.

In [ ]:
import os
import json
import time
import math
import random
import glob
import ast
import re
import subprocess
import tempfile
import urllib.request
import urllib.error

REPO_DIR = "/kaggle/working/vllm-thinking-budget"
WORK_DIR = "/kaggle/working"


def detect_gpus():
    """Return (gpu_names:list, gpu_count). Also exposes GPU_RATE_USD_S later."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip().splitlines()
        names = [l.rsplit(",", 1)[0].strip() for l in out]
        return names, len(names)
    except Exception:
        return [], 0


class VLLMServer:
    """Launches vLLM servers and exposes their ports.

    ``n_servers`` instances of ``tp_size`` GPUs each: TP=1 gives one server per
    GPU (data-parallel); TP>1 shards one model that does not fit a single GPU.

    ``self.ports`` is the list the evaluator round-robins over.
    """

    def __init__(self, model, port=8001, n_servers=1, tp_size=1,
                 repo_dir=REPO_DIR, work_dir=WORK_DIR):
        self.model = model
        self.port = port  # first server port
        self.n_servers = max(1, n_servers)
        self.ports = [port + i for i in range(self.n_servers)]
        self.repo_dir = repo_dir
        self.work_dir = work_dir
        self.processes = []
        self.gpu_names, self.gpu_count = [], 0
        self.tp_size = max(1, int(tp_size))  # TP=1: one server per GPU (data-parallel); TP>1: weights sharded over tp_size GPUs
        self.max_model_len = 32768
        self.max_num_seqs = 96
        self.kv_tokens = []   # filled from the vLLM logs after start

    # ---- fork patch: stop the token scan at the first -1 placeholder --------
    def apply_patch(self):
        path = os.path.join(self.repo_dir, "thinking_budget_processor.py")
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        content = open(path).read()
        old = ("        for i in range(self.last_scanned, end):\n"
               "            tid = tokens[i]")
        new = ("        for i in range(self.last_scanned, end):\n"
               "            tid = tokens[i]\n"
               "            if tid == -1:\n"
               "                end = i\n"
               "                break")
        if "if tid == -1:" in content:
            print("[server] \u2713 patch already applied")
        elif old in content:
            open(path, "w").write(content.replace(old, new))
            print("[server] \u2713 patch applied")
        else:
            print("[server] \u26a0 patch pattern not found \u2014 continuing anyway")

    # ---- config ------------------------------------------------------------
    def write_config(self):
        # Robust VRAM detection: sum all GPUs, handle MiB suffix / blanks
        gpu_count, total_vram_gb, gpu_names = 0, 0, []
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,memory.total",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=10,
            ).stdout.strip()
            for line in out.splitlines():
                line = line.strip()
                if not line:
                    continue
                parts = [p.strip() for p in line.rsplit(",", 1)]
                if len(parts) == 2:
                    name, mem = parts
                    gpu_names.append(name)
                    total_vram_gb += int(float(mem)) / 1024.0
                    gpu_count += 1
        except Exception as e:
            print(f"[server] nvidia-smi failed: {e}", flush=True)
        self.gpu_count = gpu_count
        self.gpu_names = gpu_names
        if gpu_count and gpu_count < self.tp_size:
            raise RuntimeError(
                f"{self.model} needs tensor-parallel-size={self.tp_size} but only "
                f"{gpu_count} GPU(s) were detected")
        usable = max(1, min(self.n_servers,
                            gpu_count // self.tp_size if gpu_count else 1))
        self.ports = self.ports[:usable]  # never launch more servers than GPU groups
        print(f"[server] detected {gpu_count} GPU(s): {gpu_names} "
              f"total={total_vram_gb:.0f}GB \u2192 {usable} server(s), TP={self.tp_size} each",
              flush=True)

        # Default sizing assumes the weights leave room for a large KV cache
        # (LFM2.5-2.6B ~5.2 GB and Qwen3-0.6B ~1.2 GB on one T4; the 8B model is
        # sharded over two T4s). Step down when VRAM per GPU is low.
        per_gpu_gb = total_vram_gb / max(1, usable * self.tp_size)
        max_model_len, util, max_num_seqs = 32768, 0.95, 265
        if per_gpu_gb > 0 and per_gpu_gb < 12:
            max_model_len, util, max_num_seqs = 16384, 0.92, 128
            print("[server] \u26a0 low VRAM per GPU \u2014 max-model-len=16384", flush=True)

        self.max_model_len = max_model_len
        self.max_num_seqs = max_num_seqs
        self.config_paths = []
        for i, port in enumerate(self.ports):
            name = "vllm-config.yaml" if i == 0 else f"vllm-config-{port}.yaml"
            path = os.path.join(self.work_dir, name)
            open(path, "w").write(
                f"model: {self.model}\n"
                f"max-model-len: {max_model_len}\n"
                f"tensor-parallel-size: {self.tp_size}\n"
                f"gpu-memory-utilization: {util}\n"
                f"max-num-seqs: {max_num_seqs}\n"
                "enable-prefix-caching: true\n"
                f"port: {port}\n"
                "logits-processors:\n"
                '- "thinking_budget_processor:ThinkingBudgetLogitsProcessor"\n'
            )
            self.config_paths.append(path)
        self.config_path = self.config_paths[0]
        print(f"[server] \u2713 wrote {len(self.config_paths)} config(s): "
              f"{[os.path.basename(p) for p in self.config_paths]}")
        print(f"[server] tp_size={self.tp_size} max-model-len={max_model_len} "
              f"max-num-seqs={max_num_seqs} util={util}")

    # ---- lifecycle ---------------------------------------------------------
    def _wait_ready(self, proc, port, log_path, timeout=600):
        start_time = time.time()
        last_report = 0
        while time.time() - start_time < timeout:
            if proc.poll() is not None:
                print(f"[server] \u274c vLLM on port {port} exited rc={proc.returncode} "
                      "\u2014 last 30 log lines:", flush=True)
                for line in open(log_path).readlines()[-30:]:
                    print("   " + line.rstrip(), flush=True)
                raise RuntimeError(
                    f"vLLM on port {port} exited rc={proc.returncode}")
            try:
                r = urllib.request.urlopen(
                    f"http://localhost:{port}/health", timeout=2)
                if r.status == 200:
                    print(f"[server] \u2713 port {port} READY after "
                          f"{time.time()-start_time:.0f}s", flush=True)
                    return
            except (urllib.error.URLError, urllib.error.HTTPError):
                pass
            elapsed = time.time() - start_time
            if elapsed - last_report >= 15:
                print(f"[server] port {port} still starting... "
                      f"{elapsed:.0f}s/{timeout}s", flush=True)
                last_report = elapsed
            time.sleep(2)
        raise RuntimeError(
            f"vLLM on port {port} not healthy in time \u2014 check {log_path}")

    def _free_gpus(self, timeout=40):
        """Wait until killed vLLM processes have released GPU memory."""
        t0 = time.time()
        while time.time() - t0 < timeout:
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=10,
                ).stdout.split()
                if all(int(float(x)) < 800 for x in out):
                    return
            except Exception:
                return
            time.sleep(2)
        print("[server] WARN: GPU memory not fully freed before launch",
              flush=True)

    @staticmethod
    def _log_has(log_path, needle):
        try:
            return needle in open(log_path, errors="ignore").read()
        except Exception:
            return False

    @staticmethod
    def _parse_kv_tokens(log_path, default=100_000):
        try:
            m = re.search(r"GPU KV cache size: ([\d,]+) tokens",
                          open(log_path, errors="ignore").read())
            if m:
                return int(m.group(1).replace(",", ""))
        except Exception:
            pass
        return default

    def start(self, timeout=600):
        print("\n" + "=" * 60, flush=True)
        print(f"[server] GPUs: {self.gpu_names or 'none detected'} "
              f"(count={self.gpu_count}) servers={len(self.ports)}", flush=True)
        print(f"[server] starting vLLM ({len(self.ports)} server(s), TP={self.tp_size} each)...", flush=True)
        print("=" * 60, flush=True)

        subprocess.run(["pkill", "-9", "-f", "vllm"], check=False)
        subprocess.run(["pkill", "-9", "-f", "VLLM::"], check=False)
        time.sleep(3)
        self._free_gpus()

        env = os.environ.copy()
        env["PYTHONPATH"] = self.repo_dir + ":" + env.get("PYTHONPATH", "")

        self.processes = []
        launched = []
        for i, port in enumerate(self.ports):
            proc_env = env.copy()
            devices = ",".join(str(i * self.tp_size + j) for j in range(self.tp_size))
            proc_env["CUDA_VISIBLE_DEVICES"] = devices  # tp_size GPUs per server
            log_path = os.path.join(self.work_dir, f"vllm-{port}.log")
            log_file = open(log_path, "w")
            p = subprocess.Popen(
                ["vllm", "serve", self.model, "--config", self.config_paths[i]],
                env=proc_env, stdout=log_file, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            self.processes.append(p)
            launched.append((p, port, log_path))
            print(f"[server] started port {port} pid={p.pid} "
                  f"(CUDA_VISIBLE_DEVICES={devices})", flush=True)
            # Stagger: let this server finish torch.compile first so the next
            # one reuses the compile cache instead of racing to write it.
            # Everything after that overlaps, so total startup is ~1 server
            # long instead of N servers long.
            if i < len(self.ports) - 1:
                t0 = time.time()
                while (time.time() - t0 < 180 and p.poll() is None
                       and not self._log_has(log_path, "torch.compile took")):
                    time.sleep(2)

        for p, port, log_path in launched:
            self._wait_ready(p, port, log_path, timeout=timeout)

        self.kv_tokens = [self._parse_kv_tokens(lp) for _, _, lp in launched]
        print(f"[server] ALL SERVERS READY: {self.ports} "
              f"KV cache tokens per server: {self.kv_tokens}", flush=True)
        self.warmup()

    def warmup(self):
        """Fire a few concurrent requests of varied length at every server so
        Triton JIT / CUDA-graph shapes are compiled BEFORE the sweep (the
        original logs showed `kernel_unified_attention` JIT during inference,
        i.e. a latency spike in the middle of the run)."""
        from concurrent.futures import ThreadPoolExecutor
        t0 = time.time()

        def one(port, n_words, think):
            body = {
                "model": self.model,
                "messages": [{"role": "user", "content":
                              "Continue: " + " ".join(str(i)
                                                      for i in range(n_words))}],
                "max_tokens": 48, "temperature": 0.7,
                "skip_special_tokens": False,
                "vllm_xargs": {"max_thinking_tokens": think},
            }
            req = urllib.request.Request(
                f"http://localhost:{port}/v1/chat/completions",
                data=json.dumps(body).encode(),
                headers={"Content-Type": "application/json"})
            try:
                urllib.request.urlopen(req, timeout=300).read()
            except Exception as e:
                print(f"[server] warmup request failed on {port}: {e}",
                      flush=True)

        shapes = [(8, 0), (64, 16), (512, 32), (2000, 48)] * 3
        jobs = [(p, n, t) for p in self.ports for n, t in shapes]
        with ThreadPoolExecutor(len(jobs)) as ex:
            list(ex.map(lambda a: one(*a), jobs))
        print(f"[server] warmup done in {time.time()-t0:.0f}s", flush=True)

    def stop(self):
        subprocess.run(["pkill", "-9", "-f", "vllm"], check=False)
        subprocess.run(["pkill", "-9", "-f", "VLLM::"], check=False)
        print("[server] vLLM stopped.")

    def setup_and_start(self):
        self.apply_patch()
        self.write_config()
        # A TP>1 model is large: allow time for the weights to download and load.
        self.start(timeout=600 if self.tp_size == 1 else 1500)

## 3. Models and thinking tokens

### 3.1 Terminology

| Term | Meaning |
|---|---|
| **Boundary tokens** | The opening and closing delimiters of the reasoning segment. The closing delimiter also marks the switch to the final answer. |
| **Thinking tokens** | Everything generated between the two delimiters. Recorded as `thinking_tokens` (re-tokenized locally; approximate). |
| **Thinking budget** | `max_thinking_tokens`, the independent variable. A logits processor counts generated thinking tokens and, at `budget − 1`, forces the closing boundary so the model must answer. |
| **Truncated trace** | Generation ended without a closing boundary (`was_truncated = True`): unfinished reasoning, never an answer. |

### 3.2 Side-by-side comparison

| | **LFM2.5-2.6B** | **Qwen3-0.6B** | **DeepSeek-R1-Distill-Llama-8B** |
|---|---|---|---|
| Developer | Liquid AI | Qwen (Alibaba) | DeepSeek |
| Architecture | Hybrid: 22 gated short-convolution blocks + 8 GQA attention blocks | Dense transformer | Llama-3.1-8B architecture, distilled from DeepSeek-R1 reasoning traces |
| Parameters | 2.6B | 0.6B | 8B |
| Reasoning mode | Pure reasoning: always thinks | Hybrid thinking / non-thinking model; thinking mode is used here | Always thinks; no native off switch |
| Context window | 128K | 32K native | 128K |
| Opening boundary | Dedicated special token | `<think>` (id 151667) | `<think>` |
| Closing boundary | Dedicated special token that switches to the response | `</think>` (id 151668) | `</think>` |
| Boundary ids | 124901 (thinking) / 124902 (response) | 151667 / 151668, cross-checked by name | Resolved by name at pre-flight (expected 128013 / 128014) |
| Who produces the opening boundary | Chat template opens the thinking segment | **The model** generates `<think>` as its first token | **The chat template**: the generation prompt ends with `<think>\n` |
| Appears in the completion | Closing boundary (opening depending on template placement) | Both `<think>` and `</think>` | Only `</think>` |
| Special-token status | Special: stripped by default unless overridden | Regular added tokens | Validate; the override is harmless |
| fp16 weights | ≈ 5.2 GB | ≈ 1.2–1.5 GB | ≈ 16 GB |
| Fits one 16 GB T4 | Yes | Yes | No |
| Serving layout | Data-parallel, one server per GPU | Data-parallel, one server per GPU | One TP = 2 server over both GPUs |
| Vendor-recommended sampling | not specified | T = 0.6, top-p 0.95, top-k 20 (thinking mode) | T ≈ 0.6 (0.5–0.7), no system prompt |

**Common protocol.** All three models receive the question as the only user message (no system prompt, no thinking-mode flag) and are sampled at `temperature = 0.7`. Holding the protocol constant is deliberate: it isolates the effect of the model and the budget. It differs from each vendor's recommended setting, which limits absolute scores, not the comparison.

### 3.3 Model notes

**LFM2.5-2.6B.** A 2.6B pure-reasoning hybrid. Its thinking segment is bounded by two dedicated special tokens, so the boundary is unambiguous and cheap for the budget processor to detect. Because both are *special* tokens, vLLM's default decoding (`skip_special_tokens=True`) would strip them from the response text; every request therefore disables that default.

**Qwen3-0.6B.** The smallest model, and the most likely to loop or thrash at long budgets, which makes it a natural stress case for the *quality of thinking* question. The model emits `<think>` itself, reasons, emits `</think>`, then answers. Because the evaluator splits on the closing boundary only, the stored `thinking_text` **begins with the literal `<think>` tag**; strip it before text analysis. The model supports a native non-thinking switch; this study does not use it, and budget 0 is produced by forcing the closing boundary instead, so that all three models are treated identically.

**DeepSeek-R1-Distill-Llama-8B.** The opening `<think>\n` is part of the **prompt**, so the completion starts *inside* the reasoning and contains only `</think>`. The splitting logic works unchanged, but any component that waits to *observe* an opening tag among the generated tokens would never see one: the budget counter must start at the first generated token, which the pilot verifies (§7). Distilled models also occasionally skip reasoning altogether (an empty `<think></think>` block), visible as rows with near-zero `thinking_tokens` at non-zero budgets.

### 3.4 What the differences imply for comparability

1. **Token counts are tokenizer-specific.** `thinking_tokens` is computed with the served model's own tokenizer (vocabularies of roughly 128K for Llama-3, 152K for Qwen3, and Liquid's own for LFM). Within a model, token counts are exact enough for curve-shaped analysis. **Between models, compare characters or words**, or re-tokenize every trace with one reference tokenizer.
2. **The budget caps generated tokens only.** For DeepSeek the opening tag is prompt text and is not counted; for Qwen3 the generated `<think>` token may count as the first thinking token. The difference is one or two tokens: negligible from 256 upward, visible only at the smallest budgets.
3. **Budget 0 yields different text per model**, because the opening tag is generated (Qwen3), pre-filled (DeepSeek) or template-opened (LFM). Level-0 rows should be inspected by eye for each model.
4. **A forced close cuts reasoning mid-sentence.** When the cap binds, the last sentence of `thinking_text` is artificial. Treat the tail of such traces as an artifact of the intervention, not model behavior.
5. **Cost scales very differently.** The 0.6B and 2.6B models are decode-bound on a T4; the 8B on a TP = 2 server holds far fewer concurrent sequences, so the same grid takes substantially longer.

## 4. Launch and pre-flight check

### Model selection

`MODEL_KEY` selects one entry of `MODEL_CONFIGS`, which holds everything that is model-specific: the Hugging Face id, the tensor-parallel size and the boundary tokens. A boundary token is given either as a fixed id or as a string that is resolved by name; when both are given they must agree.

### Why the tokenizer check must run before anything else

The budget processor resolves the boundary tags **by name**, plus the newline token it forces at `budget − 1`. If any of these does not encode to exactly **one** token in the served model's tokenizer, the processor can never force the switch, and every call silently burns its full `max_tokens` instead of respecting its budget. That failure mode is dangerous precisely because it is invisible: the run completes, the CSV fills, the curves look plausible, and none of the numbers mean anything.

The pre-flight check therefore:

1. resolves the opening and closing ids and decodes them back to strings,
2. re-encodes each string and requires **exactly one token equal to the configured id**,
3. requires `"\n"` to be a single token,
4. **aborts the run** (raises) on any failure rather than warning,
5. prints the tail of the rendered chat-template prompt, so it is visible whether the opening tag sits in the prompt (DeepSeek) or is generated by the model (Qwen3).

The tokenizer loaded here (`_tok`) is reused downstream by the evaluator to count thinking and output tokens, so no second tokenizer download is needed.

In [ ]:
# ---------------------------------------------------------------------------
# Model selection — the only place where the three models differ.
#   open_id / close_id : fixed boundary-token ids (None -> resolved by name)
#   open_str / close_str: boundary strings (used to resolve or cross-check ids)
#   tp_size            : 1 = one server per GPU (data-parallel); 2 = one server over 2 GPUs
# ---------------------------------------------------------------------------
MODEL_KEY = "qwen3-0.6b"   # "lfm2.5-2.6b" | "qwen3-0.6b" | "deepseek-r1-distill-llama-8b"

MODEL_CONFIGS = {
    "lfm2.5-2.6b": dict(
        model="LiquidAI/LFM2.5-2.6B", tp_size=1,
        open_id=124901, close_id=124902, open_str=None, close_str=None),
    "qwen3-0.6b": dict(
        model="Qwen/Qwen3-0.6B", tp_size=1,
        open_id=151667, close_id=151668, open_str="<think>", close_str="</think>"),
    "deepseek-r1-distill-llama-8b": dict(
        model="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", tp_size=2,
        open_id=None, close_id=None, open_str="<think>", close_str="</think>"),
}
cfg = MODEL_CONFIGS[MODEL_KEY]
Model = cfg["model"]
os.environ["VLLM_USE_V2_MODEL_RUNNER"] = "0"

# One server per TP group of GPUs; the evaluator round-robins over server.ports.
_gpu_names, _gpu_count = detect_gpus()
N_SERVERS = max(1, _gpu_count // cfg["tp_size"]) if _gpu_count >= 1 else 1

server = VLLMServer(model=Model, port=8001, n_servers=N_SERVERS,
                    tp_size=cfg["tp_size"])
server.setup_and_start()

# Billing/throughput knobs for live KPIs (T4 $0.164/hr = $0.000164/s).
GPU_COUNT = len(server.ports) * server.tp_size
GPU_RATE_USD_S = 0.000164 * GPU_COUNT
# Upper bound on client threads. Real concurrency is decided by the
# KV-aware admission control inside ThinkingBudgetEvaluator (<= max-num-seqs
# minus a margin per server, and never more than the KV cache can hold).
MAX_WORKERS = 64 * len(server.ports)
print(f"[run] model={Model} gpus={GPU_COUNT} servers={len(server.ports)} "
      f"tp={server.tp_size} rate=${GPU_RATE_USD_S*3600:.2f}/hr workers={MAX_WORKERS}")


# Pre-flight: the budget processor resolves the boundary tags by name, so each
# must encode to exactly one token. Abort loudly if not.

from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(Model, trust_remote_code=True)


def _resolve_boundary(tag_id, tag_str, name):
    """Fixed id if given (cross-checked against the string when both exist),
    otherwise resolved by name from the tokenizer."""
    if tag_str is not None:
        by_name = _tok.convert_tokens_to_ids(tag_str)
        unk = getattr(_tok, "unk_token_id", None)
        by_name = None if (by_name is None or by_name == unk) else by_name
        if tag_id is None:
            if by_name is None:
                raise RuntimeError(
                    f"{Model}: cannot resolve the {name} boundary {tag_str!r} "
                    "to a token id - aborting.")
            return by_name
        if by_name is not None and by_name != tag_id:
            raise RuntimeError(
                f"{Model}: configured {name} id {tag_id} disagrees with "
                f"{tag_str!r} -> {by_name} - aborting.")
    return tag_id


THINKING_TOKEN_ID = _resolve_boundary(cfg["open_id"], cfg["open_str"], "thinking")
RESPONSE_TOKEN_ID = _resolve_boundary(cfg["close_id"], cfg["close_str"], "response")
THINKING_MARKER = _tok.decode([THINKING_TOKEN_ID])
RESPONSE_MARKER = _tok.decode([RESPONSE_TOKEN_ID])

for _name, _tid, _marker in [
    ("thinking", THINKING_TOKEN_ID, THINKING_MARKER),
    ("response", RESPONSE_TOKEN_ID, RESPONSE_MARKER),
]:
    _enc = _tok.encode(_marker, add_special_tokens=False)
    _ok = len(_enc) == 1 and _enc[0] == _tid
    print(f"[run] tok {_name}: id={_tid} decoded={_marker!r} round_trip={_enc} ok={_ok}")
    if not _ok:
        raise RuntimeError(
            f"{Model}: token id {_tid} ({_name}) does not round-trip through "
            f"decode->encode as a single token ({_enc}); the thinking-budget "
            "processor and the text split would not agree - aborting.")

_nl_enc = _tok.encode("\n", add_special_tokens=False)
_nl_ok = len(_nl_enc) == 1 and _nl_enc[0] not in (None, -1)
print(f"[run] tok '\\n': id={_nl_enc[0] if _nl_ok else _nl_enc} ok={_nl_ok}")
if not _nl_ok:
    raise RuntimeError(f"'\\n' is not a single token ({_nl_enc}) - aborting.")

# Where does the opening boundary come from? In the prompt (e.g. DeepSeek) the
# completion contains only the closing tag; otherwise the model generates it.
try:
    _prompt = _tok.apply_chat_template(
        [{"role": "user", "content": "What is 2+2?"}],
        tokenize=False, add_generation_prompt=True)
    print(f"[run] chat-template tail: {_prompt[-60:]!r}")
    print(f"[run] opening boundary in prompt: {THINKING_MARKER in _prompt}")
except Exception as _e:
    print(f"[run] chat-template inspection skipped: {_e}")

print(f"[run] tokenizer OK: thinking={THINKING_MARKER!r} response={RESPONSE_MARKER!r}")

## 5. Benchmark suite

Nine benchmarks across three assumed difficulty tiers and six task types:

| Benchmark | Tier | Task type | Source |
|---|---|---|---|
| GSM8K | easy | math | `openai/gsm8k` |
| MMLU-Redux | easy | general knowledge | `edinburgh-dawg/mmlu-redux-2.0` (4 subjects) |
| IFEval | easy | instruction following | `google/IFEval` |
| MATH (full) | medium | math | `DigitalLearningGmbH/MATH-lighteval` |
| MMLU-Pro | medium | general knowledge | `TIGER-Lab/MMLU-Pro` |
| BFCL v4 `simple_python` | medium | agentic tool use | sparse clone of `ShishirPatil/gorilla` |
| AIME 2025 | hard | math | `test-time-compute/aime_2025` |
| GPQA-Diamond | hard | PhD-level science | `Idavidrein/gpqa` (gated) |
| LiveCodeBench v6 | hard | coding | `lighteval/code_generation_lite` |

MMLU-Redux mixes four subjects: high-school geography, high-school world history, miscellaneous, and professional law.

### Difficulty tiers are hypotheses, not facts

Tiers are literature-based priors; vendor-published scores exist for only a few of these benchmarks and models. Because every tier-level result inherits the labels, the analysis re-checks them empirically against each model's own low-budget accuracy and flags mismatches.

### Every grader takes the *last* match

A reasoning model writes "maybe A… no, B fits better". Taking the **first** regex match returns A — the scratch mention — instead of B, the model's actual pick. Every grader therefore scans from the end: last standalone letter, last number, last `\boxed{}`, last fenced code block, last parseable function call. Graders receive only the **answer half** of the completion (the thinking half is separated upstream), so this is defensive rather than load-bearing, but it removes a whole class of silent mis-grading.

### Deliberate simplifications

Three graders are lighter than the official protocols, and each is documented in its class:

- **IFEval** — implements about 8 of the ~25 official verifiable-instruction checkers and keeps only questions that use them. Unsupported instruction ids are skipped, not failed.
- **BFCL** — syntactic call matching instead of the official semantic AST checker.
- **LiveCodeBench** — grades on **public** test cases only, so scores are an optimistic bound.

These are acceptable for a *within-model, across-budget* comparison: every budget level passes through the same lenient grader, so the *shape* of each curve survives even though the *level* is biased. Scores are not comparable to published leaderboards.

In [ ]:
from abc import ABC, abstractmethod
from datasets import load_dataset

import subprocess
import tempfile
import os
import re
import ast
import json
import glob
import random
import math
import zlib
import base64


class Benchmark(ABC):
    """Base class — implement load() and grade() for any new benchmark.

    Every subclass declares:
      - name: str, unique identifier used in the raw results table
      - difficulty_tier: "easy" | "medium" | "hard" — a working hypothesis
        from the literature difficulty ordering, not a verified fact about
        the studied models; the analysis re-checks it empirically.
      - task_type: category label used for disaggregation (matches the target
        benchmark table: math / general_knowledge / instruction_following /
        agentic_tool_use / phd_science / coding)
    """

    name = None
    difficulty_tier = None
    task_type = None

    @abstractmethod
    def load(self, n):
        """Returns list of {"question": str, "gold": any}"""
        ...

    @abstractmethod
    def grade(self, output_text, gold):
        """Returns True/False. `output_text` is the final-answer portion only;
        the evaluator strips the thinking text upstream.
        """
        ...


def _extract_boxed(text):
    """Extract the last \\boxed{...} contents, handling nested braces."""
    idxs = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not idxs:
        return None
    start = idxs[-1] + len("\\boxed{")
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start:i - 1].strip()


# ================================================================== easy tier
class GSM8KBenchmark(Benchmark):
    """Easy / Math — GSM8K (grade-school arithmetic word problems).

    Tier: "easy" is a literature-based hypothesis — GSM8K sits near ceiling for
    modern ~2-8B reasoning models. Liquid AI publishes no LFM2.5-2.6B GSM8K
    score, so the tier is validated empirically in the analysis cell.
    """

    name = "GSM8K"
    difficulty_tier = "easy"
    task_type = "math"

    def load(self, n=30):
        ds = load_dataset(
            "openai/gsm8k",
            "main",
            split="test").select(
            range(n))
        questions = []
        for row in ds:
            gold = re.search(
                r"####\s*(-?[\d,]+\.?\d*)",
                row["answer"]).group(1).replace(
                ",",
                "")
            questions.append({"question": row["question"], "gold": gold})
        return questions

    def grade(self, output_text, gold):
        numbers = re.findall(r"-?\d[\d,]*\.?\d*", output_text)
        if not numbers:
            return False
        try:
            return abs(
                float(numbers[-1].replace(",", "")) - float(gold)) < 1e-4
        except ValueError:
            return False


class MMLURedux(Benchmark):
    """Easy / General knowledge — MMLU-Redux.

    MMLU-Redux re-annotates a subset of the MMLU test sets to fix labeling
    errors, so it shares MMLU's per-subject schema; a handful of subjects are
    mixed here for a representative sample.

    Tier: "easy" is a literature-based hypothesis — the relabeling keeps MMLU's
    difficulty profile, well below PhD-level suites like GPQA. No official
    LFM2.5-2.6B score exists.
    """

    name = "MMLU-Redux"
    difficulty_tier = "easy"
    task_type = "general_knowledge"

    DEFAULT_SUBJECTS = [
        "high_school_geography",
        "high_school_world_history",
        "miscellaneous",
        "professional_law",
    ]

    def __init__(self, subjects=None):
        self.subjects = subjects or self.DEFAULT_SUBJECTS

    def load(self, n=30):
        per_subject = max(1, math.ceil(n / len(self.subjects)))
        questions = []
        for subject in self.subjects:
            ds = load_dataset(
                "edinburgh-dawg/mmlu-redux-2.0",
                subject,
                split="test")
            ds = ds.select(range(min(per_subject, len(ds))))
            for row in ds:
                choices = "\n".join(
                    f"{chr(65 + i)}. {c}"
                    for i, c in enumerate(row["choices"])
                )
                q = (
                    f"{row['question']}\n{choices}\n"
                    "Answer with just the letter."
                )
                questions.append(
                    {"question": q, "gold": chr(65 + row["answer"])})
        return questions[:n]

    def grade(self, output_text, gold):
        # Take the LAST standalone letter: a model may name other options
        # before settling on its real pick.
        matches = re.findall(r"\b([A-D])\b", output_text.strip())
        if not matches:
            return False
        return matches[-1] == gold


class IFEvalBenchmark(Benchmark):
    """Easy / Instruction-following — IFEval (programmatically verifiable
    output constraints).

    Tier: "easy-to-medium" prior. No official LFM2.5-2.6B IFEval score exists;
    Liquid AI publishes strong numbers on a different in-house suite
    (IFStruct 85.49, Multi-IF 80.07, IFBench 59.17), which is suggestive of the
    general capability but is not the same benchmark.

    Simplification: the official grader implements ~25 verifiable-instruction
    checkers (instructions_registry.py in google-research/
    instruction_following_eval). This loader keeps only questions that use the
    subset implemented below — keyword existence/frequency, forbidden words,
    word/sentence/paragraph counts, lowercase, bullet-list count. Swap in the
    official registry for publication-grade rigor.
    """

    name = "IFEval"
    difficulty_tier = "easy"
    task_type = "instruction_following"

    _SUPPORTED_PREFIXES = (
        "keywords:existence",
        "keywords:frequency",
        "keywords:forbidden_words",
        "length_constraints:number_words",
        "length_constraints:number_sentences",
        "length_constraints:number_paragraphs",
        "change_case:english_lowercase",
        "detectable_format:number_bullet_lists",
    )

    def load(self, n=30):
        ds = load_dataset("google/IFEval", split="train")
        questions = []
        for row in ds:
            ids = row["instruction_id_list"]
            if not any(i.startswith(self._SUPPORTED_PREFIXES) for i in ids):
                continue
            questions.append({
                "question": row["prompt"],
                "gold": {"instruction_id_list": ids, "kwargs": row["kwargs"]},
            })
            if len(questions) >= n:
                break
        return questions

    def _check_one(self, text, instr_id, kwargs):
        try:
            if instr_id.startswith("keywords:existence"):
                return all(kw.lower() in text.lower()
                           for kw in kwargs.get("keywords", []))
            if instr_id.startswith("keywords:frequency"):
                kw = kwargs.get("keyword", "")
                target = kwargs.get("frequency", 0)
                rel = kwargs.get("relation", "at least")
                count = text.lower().count(kw.lower())
                if rel == "at least":
                    return count >= target
                return count <= target
            if instr_id.startswith("keywords:forbidden_words"):
                return all(kw.lower() not in text.lower()
                           for kw in kwargs.get("forbidden_words", []))
            if instr_id.startswith("length_constraints:number_words"):
                n_words = len(text.split())
                target = kwargs.get("num_words", 0)
                rel = kwargs.get("relation", "at least")
                if rel == "at least":
                    return n_words >= target
                return n_words <= target
            if instr_id.startswith("length_constraints:number_sentences"):
                n_sent = len(re.findall(r"[.!?]+", text))
                target = kwargs.get("num_sentences", 0)
                rel = kwargs.get("relation", "at least")
                if rel == "at least":
                    return n_sent >= target
                return n_sent <= target
            if instr_id.startswith("length_constraints:number_paragraphs"):
                n_para = len([p for p in text.split("\n\n") if p.strip()])
                return n_para == kwargs.get("num_paragraphs", 0)
            if instr_id.startswith("change_case:english_lowercase"):
                return text == text.lower()
            if instr_id.startswith("detectable_format:number_bullet_lists"):
                n_bullets = len(
                    re.findall(
                        r"^\s*[\*\-]\s",
                        text,
                        flags=re.MULTILINE))
                return n_bullets == kwargs.get("num_bullets", 0)
        except Exception:
            return False
        return True  # unsupported id slipped through -> don't penalize

    def grade(self, output_text, gold):
        ids = gold["instruction_id_list"]
        kwargs_list = gold["kwargs"]
        results = []
        for instr_id, kwargs in zip(ids, kwargs_list):
            if not instr_id.startswith(self._SUPPORTED_PREFIXES):
                continue
            results.append(
                self._check_one(
                    output_text,
                    instr_id,
                    kwargs or {}))
        return len(results) > 0 and all(results)


# ================================================================ medium tier
class MATHFullBenchmark(Benchmark):
    """Medium / Math — full MATH test set.

    Tier: "medium" is a literature-based hypothesis — MATH sits above GSM8K and
    below AIME in published difficulty orderings. No official LFM2.5-2.6B score
    exists.

    Dataset: `hendrycks/competition_math` was disabled on the Hub after a DMCA
    takedown. `DigitalLearningGmbH/MATH-lighteval` is a re-upload with the same
    schema (problem/level/type/solution) and a `default` config holding all
    subjects in one 5000-example test split.
    """

    name = "MATH-full"
    difficulty_tier = "medium"
    task_type = "math"

    def load(self, n=30):
        ds = load_dataset(
            "DigitalLearningGmbH/MATH-lighteval",
            "default",
            split="test",
        )
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for row in ds:
            gold = _extract_boxed(row["solution"])
            if gold is None:
                continue
            questions.append({"question": row["problem"], "gold": gold})
        return questions

    def grade(self, output_text, gold):
        pred = _extract_boxed(output_text)
        if pred is None:
            lines = output_text.strip().splitlines()
            # No \boxed{} and no text at all (e.g. a truncated call with no
            # final answer) — nothing to grade against; structurally incorrect.
            pred = lines[-1].strip() if lines else ""
        return pred.replace(" ", "") == gold.replace(" ", "")


class MMLUProBenchmark(Benchmark):
    """Medium / General knowledge — MMLU-Pro (10-option, reasoning-heavier MMLU).

    Tier: "medium" is a hypothesis; no official LFM2.5-2.6B score exists.

    Dataset: the TIGER-Lab/MMLU-Pro answer column is an *integer index* (0-9),
    not a letter; grade() converts it before comparing.
    """

    name = "MMLU-Pro"
    difficulty_tier = "medium"
    task_type = "general_knowledge"

    def load(self, n=30):
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test").select(range(n))
        questions = []
        for row in ds:
            options = row["options"]
            choices = "\n".join(
                f"{chr(65 + i)}. {c}" for i, c in enumerate(options))
            q = f"{row['question']}\n{choices}\nAnswer with just the letter."
            questions.append({"question": q, "gold": row["answer"]})
        return questions

    def grade(self, output_text, gold):
        # Last lettered mention; also skips the pronoun "I" in "I pick H".
        matches = re.findall(r"\b([A-J])\b", output_text.strip())
        if not matches:
            return False
        # gold is int 0-9 from the dataset; compare to the letter
        try:
            expected_letter = chr(65 + int(gold))
        except (TypeError, ValueError):
            expected_letter = str(gold)
        return matches[-1] == expected_letter


class BFCLv3Benchmark(Benchmark):
    """Medium / Agentic tool use — BFCL "simple" single-turn Python category.

    Tier: "medium" is a structural guess — one unambiguous function call is
    less demanding than multi-turn or parallel tool use. Liquid AI's published
    BFCLv4 = 56.88 covers the full multi-category suite (multi-turn, parallel,
    irrelevance detection), not this easier subset, so it is not a valid
    expectation here.

    Dataset: the gorilla repo moved to BFCL v4 and split "simple" by language,
    so this loads BFCL_v4_simple_python.json — same id/question/function schema
    and ground_truth format as the old v3 file, but not a byte-identical
    question set.

    Simplification: the official grader (`bfcl.eval_checker`) does semantic AST
    matching against a possible-answer set with type/value constraints. This is
    a lighter check — parse the call, match the function name, and require every
    required argument's value to be among the acceptable values.
    """

    name = "BFCL-v3-simple"
    difficulty_tier = "medium"
    task_type = "agentic_tool_use"

    REPO_DIR = "/kaggle/working/gorilla"
    QUESTION_GLOB = "**/BFCL_v4_simple_python.json"
    ANSWER_GLOB = "**/possible_answer/BFCL_v4_simple_python.json"

    def _ensure_repo(self):
        if not os.path.exists(self.REPO_DIR):
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/ShishirPatil/gorilla.git", self.REPO_DIR],
                check=False,
            )

    def load(self, n=30):
        self._ensure_repo()
        q_files = glob.glob(
            os.path.join(
                self.REPO_DIR,
                self.QUESTION_GLOB),
            recursive=True)
        a_files = glob.glob(
            os.path.join(
                self.REPO_DIR,
                self.ANSWER_GLOB),
            recursive=True)
        if not q_files or not a_files:
            raise FileNotFoundError(
                "BFCL_v3_simple.json / possible_answer file not found under "
                f"{self.REPO_DIR} — the gorilla repo layout may have changed; "
                "update QUESTION_GLOB/ANSWER_GLOB."
            )

        def read_jsonl(path):
            with open(path) as f:
                return [json.loads(line) for line in f if line.strip()]

        questions_raw = read_jsonl(q_files[0])[:n]
        answers_raw = {row["id"]: row["ground_truth"]
                       for row in read_jsonl(a_files[0])}

        questions = []
        for row in questions_raw:
            qid = row["id"]
            if qid not in answers_raw:
                continue
            turns = row["question"][0] if isinstance(
                row["question"][0], list) else row["question"]
            user_msg = next(
                (t["content"] for t in turns if t["role"] == "user"),
                turns[0]["content"])
            funcs_desc = json.dumps(row["function"], indent=2)
            prompt = (
                f"You can call the following function(s):\n{funcs_desc}\n\n"
                f"User request: {user_msg}\n\n"
                "Respond with exactly one Python-style function call, "
                "e.g. func_name(arg1=value1, arg2=value2)."
            )
            questions.append({"question": prompt, "gold": answers_raw[qid]})
        return questions

    def grade(self, output_text, gold):
        # gold: list like [{"func_name": {"arg": ["acceptable", "values"]}}]
        #
        # Scan call candidates from the END backwards, accepting the first
        # that parses as Python and matches a known function name with
        # acceptable argument values — defensive against scratch calls left in
        # the text ("I'll call get_x(...) actually no, get_y(...)").
        text = output_text.strip()
        candidates = list(re.finditer(r"(\w+)\s*\((.*?)\)", text, flags=re.DOTALL))
        if not candidates:
            # fallback: old greedy single-span match, for calls containing
            # literal nested parentheses inside a string argument.
            m = re.search(r"(\w+)\s*\((.*)\)", text, flags=re.DOTALL)
            candidates = [m] if m else []

        for match in reversed(candidates):
            called_name, args_str = match.group(1), match.group(2)
            gold_entry = next(
                (entry[called_name] for entry in gold if called_name in entry),
                None)
            if gold_entry is None:
                continue
            try:
                call_ast = ast.parse(f"f({args_str})").body[0].value
                called_kwargs = {
                    kw.arg: ast.literal_eval(kw.value)
                    for kw in call_ast.keywords}
            except Exception:
                continue
            ok = all(
                arg_name in called_kwargs
                and str(called_kwargs[arg_name]) in [str(v) for v in acceptable]
                for arg_name, acceptable in gold_entry.items()
            )
            if ok:
                return True
        return False


# ================================================================== hard tier
class AIME2025Benchmark(Benchmark):
    """Hard / Math — AIME 2025 (competition problems, integer answers).

    Tier: "hard", anchored to a real number — Liquid AI publishes AIME25 =
    51.87 for LFM2.5-2.6B ("Deploy Agents Everywhere" blog post / HF model
    card). AIME 2025 rather than 2024 lowers contamination risk.
    """

    name = "AIME2025"
    difficulty_tier = "hard"
    task_type = "math"

    def load(self, n=30):
        # NOTE: HuggingFaceH4/aime_2025 no longer resolves on the Hub.
        # test-time-compute/aime_2025 is the current replacement: single
        # "test" split, 30 problems (AIME 2025 I + II combined), fields
        # question/answer instead of problem/answer.
        ds = load_dataset("test-time-compute/aime_2025", split="test")
        ds = ds.select(range(min(n, 30, len(ds))))
        question_key = "question" if "question" in ds.column_names else "problem"
        questions = []
        for row in ds:
            questions.append({
                "question": row[question_key],
                "gold": str(row["answer"]).strip(),
            })
        return questions

    def grade(self, output_text, gold):
        numbers = re.findall(r"-?\d+", output_text)
        if not numbers:
            return False
        return numbers[-1].strip() == gold


class GPQADiamondBenchmark(Benchmark):
    """Hard / PhD-level science — GPQA-Diamond.

    Tier: "hard" is a literature-based hypothesis — PhD-level and adversarially
    filtered against search. No official LFM2.5-2.6B score exists.

    Access: gated on HF — accept the terms at
    huggingface.co/datasets/Idavidrein/gpqa and set HF_TOKEN before .load().

    Loading: the raw file always lists the correct answer first, so options are
    shuffled per question with a fixed per-question seed — removes the
    positional artifact while keeping the question set reproducible.
    """

    name = "GPQA-Diamond"
    difficulty_tier = "hard"
    task_type = "phd_science"

    def load(self, n=30):
        ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for i, row in enumerate(ds):
            options = [
                row["Correct Answer"],
                row["Incorrect Answer 1"],
                row["Incorrect Answer 2"],
                row["Incorrect Answer 3"],
            ]
            rng = random.Random(i)  # deterministic per-question shuffle
            order = list(range(4))
            rng.shuffle(order)
            shuffled = [options[j] for j in order]
            gold_letter = chr(65 + order.index(0))
            choices = "\n".join(
                f"{chr(65 + i)}. {c}" for i, c in enumerate(shuffled))
            q = f"{row['Question']}\n{choices}\nAnswer with just the letter."
            questions.append({"question": q, "gold": gold_letter})
        return questions

    def grade(self, output_text, gold):
        # Last standalone letter, as in MMLURedux.grade() above.
        matches = re.findall(r"\b([A-D])\b", output_text.strip())
        if not matches:
            return False
        return matches[-1] == gold


class LiveCodeBenchV6Benchmark(Benchmark):
    """Hard / Coding — LiveCodeBench v6.

    Tier: "hard", anchored to a real number — Liquid AI publishes
    LiveCodeBench-v6 = 59.41 for LFM2.5-2.6B.

    Simplification: grading uses `public_test_cases` only (stdin/stdout pairs,
    run in a subprocess with a timeout). `private_test_cases` are
    zlib+base64+pickle-encoded and are not decoded here, so scores are an
    optimistic bound relative to the official public+private protocol. A
    problem counts as correct only if every public test case passes.
    """

    name = "LiveCodeBench-v6"
    difficulty_tier = "hard"
    task_type = "coding"
    TIME_LIMIT_S = 6

    def load(self, n=30):
        # NOTE: livecodebench/code_generation_lite uses a Python loading
        # script (with a version_tag= kwarg), which recent `datasets`
        # versions refuse to run ("Dataset scripts are no longer
        # supported"). lighteval/code_generation_lite is a parquet mirror
        # with the identical row schema, using ordinary config names
        # (e.g. "release_v6") instead of a version_tag kwarg.
        ds = load_dataset(
            "lighteval/code_generation_lite",
            "release_v6",
            split="test",
        )
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for row in ds:
            try:
                tests = json.loads(row["public_test_cases"])
            except Exception:
                tests = []
            if not tests:
                continue
            prompt = (
                f"{row['question_content']}\n\n"
                f"Starter code (if any):\n{row.get('starter_code', '')}\n\n"
                "Write a complete Python solution that reads input from stdin "
                "and writes output to stdout. Return the code in a single "
                "```python fenced block."
            )
            questions.append({"question": prompt, "gold": tests})
        return questions

    @staticmethod
    def _extract_code(text):
        # Take the LAST fenced block: a model may draft scratch snippets
        # before its final version.
        blocks = re.findall(r"```python\s*(.*?)```", text, flags=re.DOTALL)
        if not blocks:
            blocks = re.findall(r"```\s*(.*?)```", text, flags=re.DOTALL)
        return blocks[-1] if blocks else text

    def grade(self, output_text, gold):
        code = self._extract_code(output_text)
        for case in gold:
            try:
                with tempfile.NamedTemporaryFile(
                    "w", suffix=".py", delete=False
                ) as f:
                    f.write(code)
                    path = f.name
                result = subprocess.run(
                    ["python3", path],
                    input=case.get("input", ""),
                    capture_output=True,
                    text=True,
                    timeout=self.TIME_LIMIT_S,
                )
                if result.stdout.strip() != case.get("output", "").strip():
                    return False
            except Exception:
                return False
            finally:
                try:
                    os.remove(path)
                except OSError:
                    pass
        return True

## 6. Data collection

One row per model call, labeled with model / benchmark / tier / task type / compute level / sample, so every number downstream can be traced back to the calls that produced it.

### 6.1 Request protocol

Every call is a single chat completion with:

- messages: the question as the only `user` message;
- `temperature = 0.7`;
- `max_tokens = effective_max_tokens = min( max(2000, budget + 800), max_model_len − 256 )` — the budget plus a margin for the answer, so the model is never cut off before it can use its answer margin; the outer `min` keeps prompt plus output inside the context window;
- `extra_body = {"vllm_xargs": {"max_thinking_tokens": budget}, "skip_special_tokens": False}`.

### 6.2 Keeping the thinking/response boundary alive

vLLM's OpenAI-compatible server decodes with `skip_special_tokens=True` by default. For models whose boundary tags are special tokens, that would delete the boundary before the evaluator sees it, so every request disables it — for all three models, for uniformity. The completion text is then handled by four branches:

| Branch | Condition | Result |
|---|---|---|
| 1 | The server returned a separate `reasoning_content` field (only when a vLLM reasoning parser is enabled; **off by default**) | thinking = that field, output = `content` |
| 2 | The closing marker occurs in the text | `thinking, output = text.split(marker, 1)` — split at the **first** occurrence |
| 3 | No marker, but text exists | whole text stored as `thinking`, `output = ""`, `was_truncated = True` |
| 4 | Empty text | `was_truncated = (tokens_used ≥ max_tokens)` |

If a reasoning parser is ever enabled, check the field name the installed vLLM version returns.

**What `thinking_text` looks like per model after branch 2**

| Model | Completion | Stored `thinking_text` |
|---|---|---|
| LFM2.5-2.6B | `[opening?] reasoning … [response] answer` | Reasoning; may carry the opening token depending on where the template places it |
| Qwen3-0.6B | `<think>\n reasoning … \n</think>\n\n answer` | Starts with `<think>`; strip before analysis |
| DeepSeek-R1-Distill-Llama-8B | `reasoning … \n</think>\n\n answer` | Clean; the opening tag was in the prompt |

### 6.3 Truncation semantics

If no marker ever appears but text was produced, generation hit `max_tokens` **mid-thought**. That text is unfinished reasoning, not an answer, so it is stored as `thinking` with an empty answer and `was_truncated = True`. The grader then scores it incorrect — structurally right, since the model never answered. Keeping the flag separate lets the analysis distinguish *got it wrong* from *ran out of budget*; the latter is the textbook underthinking signature.

### 6.4 Separate thinking and output token counts

The OpenAI-compatible API returns **one** aggregate `completion_tokens`, so per-half counts are obtained by **re-encoding each half locally** with the served model's tokenizer. This is an **approximation**: re-tokenizing text is not bit-identical to slicing the original token stream, because merges at the boundary can differ by a token or two. It is close enough for curve-shaped analysis and is what makes *long think, short answer* distinguishable from the reverse. Do not read `thinking_tokens` as an exact count.

### 6.5 Concurrency, checkpointing and monitoring

- **KV-aware admission control.** Each request is charged an expected average KV footprint of `500 + 0.65 × budget` tokens and is admitted only to a server whose estimated load stays under 85 % of that server's KV capacity, up to a per-server sequence limit. A server with nothing in flight always admits, so an oversized request can never deadlock. This keeps every GPU busy without letting a pile of 8K-thinking requests overflow the cache, which would force vLLM to preempt and recompute sequences.
- **Shuffled scheduling** (fixed seed) so in-flight requests sit at different progress points.
- **Round-robin over clients** balances the data-parallel servers; HTTP timeout 7,200 s, no SDK retries, three explicit back-off attempts for transient errors only.
- **Append-only checkpoint** every 50 rows or 60 s, flushed with `fsync`. The result list is never held in RAM and the CSV is never rewritten. On resume only the key columns (`benchmark`, `question_id`, `compute_level`, `sample_index`) are loaded and completed calls are skipped; a partly written last row is tolerated; a checkpoint written by a different model is rejected.
- **Heartbeat every 60 s:** calls done and failed, ETA, live generation tokens/s per server, running and waiting requests, KV usage, preemptions, `/health`, running accuracy and the response-marker hit rate.
- **Failure policy.** Failed calls are not recorded and are retried on the next run; the sweep aborts cleanly after 20 consecutive failures with progress saved.
- **Marker hit rate as a canary.** If it drops mid-run, a server restarted without the `skip_special_tokens` override, or budgets are too tight relative to the answer margin.

### 6.6 Output schema (one row per call)

| Column | Meaning |
|---|---|
| `question_id`, `benchmark`, `difficulty_tier`, `task_type` | Which item, and its a-priori tier and type |
| `model` | The served model id |
| `compute_level` | Thinking budget (0 … 8192) |
| `sample_index` | Sample number 0–4 at temperature 0.7 |
| `thinking_text` | The reasoning half of the completion |
| `raw_model_output` | The **answer half** after the split (not the unsplit model text) |
| `gold`, `is_correct` | Reference answer and the lenient grader's verdict |
| `tokens_used` | Exact `completion_tokens` from the API |
| `thinking_tokens`, `output_tokens` | Local re-tokenization of each half (approximate; tokenizer-specific) |
| `hit_response_marker` | The closing marker was seen |
| `was_truncated` | Generation ran out of tokens mid-thought |
| `hit_marker_empty_output` | Marker seen but the answer is empty |
| `latency_s` | Wall time of the call (depends on concurrency as well as on the model) |

In [ ]:
import os
import re
import time
import queue
import threading
import urllib.request
import pandas as pd
import openai
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


class _LoadBalancer:
    """KV-cache-aware admission control across the data-parallel servers.

    Each request is charged an *expected average KV footprint* (tokens) and is
    only admitted to a server whose estimated load stays under that server's
    KV capacity. This keeps every GPU busy (many short requests in flight)
    without letting a pile of 8k-thinking requests overflow the KV cache, which
    is what makes vLLM preempt/recompute sequences and look "stuck".
    A server with nothing in flight always admits (so an oversized request can
    never deadlock).
    """

    def __init__(self, cap_tokens, cap_seqs):
        self.cap_tokens = list(cap_tokens)
        self.cap_seqs = cap_seqs
        self.load = [0.0] * len(self.cap_tokens)
        self.seqs = [0] * len(self.cap_tokens)
        self.cond = threading.Condition()
        self.stop = False

    def acquire(self, cost):
        with self.cond:
            while not self.stop:
                best = None
                for i in range(len(self.load)):
                    if self.seqs[i] >= self.cap_seqs:
                        continue
                    fits = self.seqs[i] == 0 or \
                        self.load[i] + cost <= self.cap_tokens[i]
                    if fits:
                        frac = self.load[i] / self.cap_tokens[i]
                        if best is None or frac < best[0]:
                            best = (frac, i)
                if best is not None:
                    i = best[1]
                    self.load[i] += cost
                    self.seqs[i] += 1
                    return i
                self.cond.wait(timeout=5)
            return None

    def release(self, i, cost):
        with self.cond:
            self.load[i] -= cost
            self.seqs[i] -= 1
            self.cond.notify_all()

    def shutdown(self):
        with self.cond:
            self.stop = True
            self.cond.notify_all()


class ThinkingBudgetEvaluator:
    """Concurrent data collector with checkpoint/resume and live progress.

    Speed/robustness changes vs. the first version:
      * KV-aware admission control (see _LoadBalancer) instead of a fixed
        16 workers/server -> the GPUs are actually saturated.
      * Longest-budget-first scheduling -> no long straggler tail at the end.
      * OpenAI client timeout raised (default 600 s killed and *re-sent*
        every long generation) and retries handled here, not by the SDK.
      * Append-only checkpoint (old code rewrote the whole multi-hundred-MB
        CSV every 25 rows) and no full result list kept in RAM.
      * O(1) running statistics (old code re-scanned every row per completion).
      * Heartbeat every 60 s with per-server load + /health, so a stall is
        visible instead of silent.
    """

    TRANSIENT = (openai.APIConnectionError, openai.APITimeoutError,
                 openai.InternalServerError, openai.RateLimitError)
    KEY_COLS = ["benchmark", "question_id", "compute_level", "sample_index"]

    def __init__(self, server, max_workers=192,
                 checkpoint="/kaggle/working/h1_raw_results.csv",
                 tokenizer=None):

        self.server = server
        self.ports = server.ports

        self.clients = [
            OpenAI(
                base_url=f"http://localhost:{p}/v1",
                api_key="dummy",
                timeout=openai.Timeout(7200.0, connect=30.0),
                max_retries=0,
            )
            for p in self.ports
        ]

        self.max_workers = max_workers
        self.checkpoint = checkpoint
        self.done = set()
        self._cols = None
        self._n_saved = 0

        self.tokenizer = (
            tokenizer if tokenizer is not None else globals().get("_tok")
        )
        self._response_marker = globals().get("RESPONSE_MARKER")

        if self.tokenizer is None:
            raise RuntimeError(
                "No tokenizer available for thinking/output token counting — "
                "run the launch cell first (it defines the global `_tok`), or "
                "pass tokenizer=... explicitly."
            )

        # ---- model length / capacity ------------------------------------
        self.max_model_len = int(getattr(server, "max_model_len", 0) or 0)
        if not self.max_model_len:
            self.max_model_len = 32768
            try:
                for line in open(os.path.join(WORK_DIR, "vllm-config.yaml")):
                    if line.startswith("max-model-len:"):
                        self.max_model_len = int(line.split(":")[1].strip())
            except Exception:
                pass
        self.max_output_tokens = self.max_model_len - 256

        self.max_num_seqs = int(getattr(server, "max_num_seqs", 64) or 64)
        kv = list(getattr(server, "kv_tokens", []) or [])
        kv += [100_000] * (len(self.ports) - len(kv))
        self.kv_tokens = kv[:len(self.ports)]
        self.cap_tokens = [0.85 * k for k in self.kv_tokens]
        self.cap_seqs = max(
            1, min(self.max_num_seqs - 4, self.max_workers // len(self.ports)))

        print(
            f"[evaluator] max_model_len={self.max_model_len} "
            f"(output capped at {self.max_output_tokens}) | "
            f"per-server: KV={self.kv_tokens} tokens, "
            f"max {self.cap_seqs} in flight",
            flush=True,
        )

        self._load_checkpoint()

    # =============================================================
    # CHECKPOINT (append-only)
    # =============================================================

    def _load_checkpoint(self):
        if not (os.path.exists(self.checkpoint)
                and os.path.getsize(self.checkpoint) > 0):
            return
        try:
            self._cols = pd.read_csv(self.checkpoint, nrows=0).columns.tolist()
            keys = pd.read_csv(self.checkpoint, usecols=self.KEY_COLS)
        except Exception as e:
            # e.g. the session was killed mid-append and the last row is cut
            print(f"[resume] ⚠ checkpoint partly unreadable ({e}); "
                  "retrying with bad-line skipping", flush=True)
            keys = pd.read_csv(self.checkpoint, usecols=self.KEY_COLS,
                               engine="python", on_bad_lines="skip")
            self._cols = pd.read_csv(
                self.checkpoint, nrows=0, engine="python").columns.tolist()
        # A checkpoint written by another model must never be resumed here.
        try:
            if "model" in (self._cols or []):
                seen = pd.read_csv(self.checkpoint, usecols=["model"]
                                   )["model"].dropna().unique()
                other = [m for m in seen if m != self.server.model]
                if other:
                    raise RuntimeError(
                        f"checkpoint {self.checkpoint} contains rows from "
                        f"{other}, but the served model is {self.server.model}. "
                        "Use a fresh session or set RESET_CHECKPOINTS = True.")
        except RuntimeError:
            raise
        except Exception:
            pass
        self._n_saved = len(keys)
        self.done = set(zip(
            keys["benchmark"].tolist(),
            keys["question_id"].tolist(),
            keys["compute_level"].tolist(),
            keys["sample_index"].tolist(),
        ))
        print(f"[resume] {len(self.done)} calls already in checkpoint "
              f"— skipping them", flush=True)

    def _append(self, rows):
        if not rows:
            return
        cols = self._cols or list(rows[0].keys())
        df = pd.DataFrame(rows, columns=cols)
        new_file = (not os.path.exists(self.checkpoint)
                    or os.path.getsize(self.checkpoint) == 0)
        with open(self.checkpoint, "a", newline="", encoding="utf-8") as f:
            df.to_csv(f, index=False, header=new_file)
            f.flush()
            os.fsync(f.fileno())
        self._cols = cols
        self._n_saved += len(rows)

    # =============================================================
    # SINGLE REQUEST
    # =============================================================

    def _create(self, server_idx, **kw):
        last = None
        for attempt in range(3):
            try:
                return self.clients[server_idx].chat.completions.create(**kw)
            except self.TRANSIENT as e:
                last = e
                time.sleep(5 * (attempt + 1))
        raise last

    def query(self, question, budget, max_tokens, temperature=0.7,
              server_idx=0):

        t0 = time.time()

        response = self._create(
            server_idx,
            model=self.server.model,
            messages=[{"role": "user", "content": question}],
            max_tokens=max_tokens,
            temperature=temperature,
            extra_body={
                "vllm_xargs": {"max_thinking_tokens": budget},
                "skip_special_tokens": False,
            },
        )

        elapsed = time.time() - t0

        full_text = response.choices[0].message.content or ""
        tokens_used = response.usage.completion_tokens

        reasoning_text = getattr(
            response.choices[0].message, "reasoning_content", None) or ""

        if reasoning_text:
            thinking = str(reasoning_text).strip()
            output = full_text.strip()
            was_truncated = False
            hit_marker = True

        elif self._response_marker and self._response_marker in full_text:
            thinking, output = full_text.split(self._response_marker, 1)
            thinking = thinking.strip()
            output = output.strip()
            was_truncated = False
            hit_marker = True

        elif full_text.strip():
            thinking = full_text.strip()
            output = ""
            was_truncated = True
            hit_marker = False

        else:
            thinking = ""
            output = ""
            was_truncated = tokens_used >= max_tokens
            hit_marker = False

        thinking_tokens = (
            len(self.tokenizer.encode(thinking, add_special_tokens=False))
            if thinking else 0
        )
        output_tokens = (
            len(self.tokenizer.encode(output, add_special_tokens=False))
            if output else 0
        )

        return (output, thinking, tokens_used, thinking_tokens,
                output_tokens, hit_marker, elapsed, was_truncated)

    # =============================================================
    # ONE EVALUATION
    # =============================================================

    def _call_one(self, q, bench, qi, budget, sample_idx,
                  max_tokens, temperature, answer_margin, server_idx):

        # prompt + output <= max_model_len
        effective_max_tokens = min(
            max(max_tokens, budget + answer_margin),
            self.max_output_tokens
        )

        (output, thinking, tokens, thinking_tokens, output_tokens,
         hit_marker, elapsed, was_truncated) = self.query(
            q["question"], budget, effective_max_tokens, temperature,
            server_idx
        )

        return {
            "model": self.server.model,
            "question_id": f"{bench.name}_{qi}",
            "benchmark": bench.name,
            "difficulty_tier": bench.difficulty_tier,
            "task_type": bench.task_type,
            "compute_level": budget,
            "sample_index": sample_idx,
            "raw_model_output": output,
            "thinking_text": thinking,
            "gold": q["gold"],
            "is_correct": bench.grade(output, q["gold"]),
            "tokens_used": tokens,
            "thinking_tokens": thinking_tokens,
            "output_tokens": output_tokens,
            "hit_response_marker": hit_marker,
            "latency_s": elapsed,
            "was_truncated": was_truncated,
            "hit_marker_empty_output": hit_marker and not output.strip(),
        }

    # =============================================================
    # HELPERS
    # =============================================================

    @staticmethod
    def _cost(budget):
        """Expected *average* KV tokens a request holds while it runs."""
        return 500.0 + 0.65 * budget

    def _health(self):
        out = []
        for p in self.ports:
            try:
                urllib.request.urlopen(
                    f"http://localhost:{p}/health", timeout=3)
                out.append("ok")
            except Exception:
                out.append("DOWN")
        return out

    def _metrics(self):
        """Real server-side numbers from vLLM's /metrics endpoint."""
        names = {"gen": "vllm:generation_tokens_total",
                 "preempt": "vllm:num_preemptions_total",
                 "run": "vllm:num_requests_running",
                 "wait": "vllm:num_requests_waiting",
                 "kv": "vllm:kv_cache_usage_perc"}
        out = []
        for p in self.ports:
            d = {}
            try:
                txt = urllib.request.urlopen(
                    f"http://localhost:{p}/metrics", timeout=3
                ).read().decode()
                for k, n in names.items():
                    m = re.findall(
                        r"^" + re.escape(n) + r"(?:\{[^}]*\})?\s+([0-9.eE+-]+)",
                        txt, re.M)
                    d[k] = sum(float(x) for x in m) if m else None
            except Exception:
                pass
            out.append(d)
        return out

    # =============================================================
    # COLLECT
    # =============================================================

    def collect(
        self,
        benchmarks,
        compute_levels,
        n_questions=30,
        n_samples=5,
        max_tokens=2000,
        temperature=0.7,
        answer_margin=800,
        schedule="shuffled",
    ):

        all_questions = {b.name: b.load(n_questions) for b in benchmarks}
        for b in benchmarks:
            print(f"[data] {b.name} -> {len(all_questions[b.name])} questions",
                  flush=True)

        # ---- build pending tasks ---------------------------------------
        tasks = []
        for b in benchmarks:
            for qi, q in enumerate(all_questions[b.name]):
                for budget in compute_levels:
                    for si in range(n_samples):
                        key = (b.name, f"{b.name}_{qi}", budget, si)
                        if key in self.done:
                            continue
                        tasks.append((q, b, qi, budget, si))

        total = len(tasks)
        if total == 0:
            print("[collect] nothing to do — all calls already done",
                  flush=True)
            return (pd.read_csv(self.checkpoint)
                    if os.path.exists(self.checkpoint) else pd.DataFrame())

        # Longest budgets first (stable sort): the big 8192/4096 jobs start
        # immediately and the run ends on short jobs, not on a lone straggler.
        # "shuffled" (default): budgets are mixed, so in-flight requests are at
        # different progress points and the *average* KV estimate holds.
        # "longest_first" starts all long jobs together; they then grow in
        # lockstep and overflow the KV cache at the same moment (preemption).
        if schedule == "longest_first":
            tasks.sort(key=lambda t: -t[3])
        elif schedule == "shuffled":
            import random
            random.Random(0).shuffle(tasks)

        total_cost = sum(self._cost(t[3]) for t in tasks)

        print(f"[collect] {total} pending calls, "
              f"<= {self.cap_seqs} in flight per server "
              f"({len(self.ports)} servers), schedule={schedule}",
              flush=True)

        lb = _LoadBalancer(self.cap_tokens, self.cap_seqs)
        resq = queue.Queue()
        pool = ThreadPoolExecutor(
            max_workers=max(self.max_workers, self.cap_seqs * len(self.ports)))

        def producer():
            for t in tasks:
                cost = self._cost(t[3])
                idx = lb.acquire(cost)
                if idx is None:
                    return
                try:
                    fut = pool.submit(
                        self._call_one, *t,
                        max_tokens, temperature, answer_margin, idx)
                except RuntimeError:          # pool already shut down
                    lb.release(idx, cost)
                    return
                fut.add_done_callback(
                    lambda f, idx=idx, cost=cost, t=t: (
                        lb.release(idx, cost), resq.put((f, t, cost))))

        threading.Thread(target=producer, daemon=True).start()

        # ---- consume results -------------------------------------------
        started = time.time()
        tokens = done = failed = consec_fail = 0
        correct = marker_hits = 0
        done_cost = 0.0
        pending = []
        last_flush = last_beat = time.time()
        prev_m, prev_t = self._metrics(), time.time()
        err_seen = set()

        try:
            with tqdm(total=total, desc="Inference", unit="req",
                      dynamic_ncols=True) as pbar:

                while done + failed < total:
                    try:
                        fut, t, cost = resq.get(timeout=15)
                    except queue.Empty:
                        fut = None

                    if fut is not None:
                        try:
                            row = fut.result()
                        except Exception as e:
                            failed += 1
                            consec_fail += 1
                            pbar.update(1)
                            sig = f"{type(e).__name__}: {str(e)[:160]}"
                            if sig not in err_seen and len(err_seen) < 10:
                                err_seen.add(sig)
                                print(f"\n[ERROR] {t[1].name} q{t[2]} "
                                      f"budget={t[3]}: {sig}", flush=True)
                        else:
                            consec_fail = 0
                            pending.append(row)
                            tokens += row["tokens_used"]
                            done += 1
                            done_cost += cost
                            correct += bool(row["is_correct"])
                            marker_hits += bool(row["hit_response_marker"])
                            pbar.update(1)

                    now = time.time()
                    elapsed = max(now - started, 1e-3)

                    if pending and (len(pending) >= 50
                                    or now - last_flush >= 60):
                        self._append(pending)
                        pending = []
                        last_flush = now

                    if now - last_beat >= 60:
                        last_beat = now
                        cur_m = self._metrics()
                        dt = max(now - prev_t, 1e-3)
                        gen = []
                        for c, p in zip(cur_m, prev_m):
                            g1, g0 = c.get("gen"), p.get("gen")
                            gen.append(round((g1 - g0) / dt)
                                       if g1 is not None and g0 is not None
                                       else None)
                        prev_m, prev_t = cur_m, now
                        eta = ("n/a" if done < 50 else
                               f"{elapsed*(total_cost-done_cost)/max(done_cost,1.0)/60:.0f}min")
                        n_ok = max(done, 1)
                        print(
                            f"\n[collect] {done}/{total} done, {failed} failed"
                            f" | elapsed={elapsed/60:.1f}min ETA~{eta}"
                            f" | LIVE gen tok/s per server={gen}"
                            f" running={[m.get('run') for m in cur_m]}"
                            f" waiting={[m.get('wait') for m in cur_m]}"
                            f" kv={[None if m.get('kv') is None else round(m['kv'], 2) for m in cur_m]}"
                            f" preemptions={[m.get('preempt') for m in cur_m]}"
                            f" health={self._health()}"
                            f" | acc={correct/n_ok:.2f}"
                            f" marker_hit={marker_hits/n_ok:.2f}",
                            flush=True)

                    if done:
                        pbar.set_postfix(
                            tok_s=f"{tokens/elapsed:.0f}",
                            fly=sum(lb.seqs),
                            fail=failed,
                            refresh=False)

                    if consec_fail >= 20:
                        raise RuntimeError(
                            "20 consecutive request failures — the vLLM "
                            f"servers look dead (health={self._health()}). "
                            "Progress is saved; restart the servers and "
                            "re-run this cell to resume.")
        finally:
            lb.shutdown()
            pool.shutdown(wait=False, cancel_futures=True)
            self._append(pending)

        print(f"\n[collect] DONE: {done} new rows, {failed} failed "
              f"(failed calls are simply re-tried on the next run) "
              f"-> {self.checkpoint} ({self._n_saved} rows total)",
              flush=True)

        return pd.read_csv(self.checkpoint)

## 7. Pilot

One question per benchmark, one sample, three far-apart budgets `[0, 256, 8192]` — **27 calls**. It is the cheapest available guard against a full run of plausible-looking garbage: a confirmation that the budget knob actually changes behavior *before* the sweep is started. Three conditions must hold for **every** model; if any fails, stop and fix it here:

1. **`marker_hit_rate` ≈ 1.0** (above 0.9 as a hard minimum) — the thinking/response split is working. A low rate means the special-token override did not take effect, the boundary ids are wrong, or budgets are too tight relative to the answer margin.
2. **`thinking_tokens` grows with the budget** — the cap is binding. For DeepSeek this is also the test that the budget counter starts at the first generated token.
3. **`output_tokens` stays roughly flat** — the budget is being spent on reasoning, not leaking into a longer final answer.

The pilot also confirms that all nine benchmarks load, and prints mean total, thinking and output tokens per budget together with accuracy by benchmark and tier. Level-0 rows should be read by eye (§3.4).

Setting `RESET_CHECKPOINTS = True` in the next cell wipes accumulated checkpoints and starts from zero; leave it `False` to resume.

In [7]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Ignoring part of dataset_info from the dataset card.*"
)

In [8]:
# Set True to wipe accumulated results and start fresh with the current
# benchmarks/COMPUTE_LEVELS. Leave False to resume normally.
RESET_CHECKPOINTS = False

if RESET_CHECKPOINTS:
    for _f in ["/kaggle/working/h2_raw_results.csv",
               "/kaggle/working/h2_pilot_results.csv"]:
        if os.path.exists(_f):
            os.remove(_f)
            print(f"[reset] removed {_f}")
    print("[reset] checkpoints cleared — next run starts from zero")
else:
    print("[reset] RESET_CHECKPOINTS=False — resuming from any existing checkpoint")

[reset] RESET_CHECKPOINTS=False — resuming from any existing checkpoint


In [9]:
benchmarks = [
    GSM8KBenchmark(),            # easy / math
    MMLURedux(),                 # easy / general_knowledge
    IFEvalBenchmark(),           # easy / instruction_following
    MATHFullBenchmark(),         # medium / math
    MMLUProBenchmark(),          # medium / general_knowledge
    BFCLv3Benchmark(),           # medium / agentic_tool_use
    AIME2025Benchmark(),         # hard / math
    GPQADiamondBenchmark(),      # hard / phd_science
    LiveCodeBenchV6Benchmark(),  # hard / coding
]

PILOT_LEVELS = [0,256, 8192]

evaluator = ThinkingBudgetEvaluator(server, max_workers=MAX_WORKERS,
                                    checkpoint="/kaggle/working/h2_pilot_results.csv")
pilot_df = evaluator.collect(
    benchmarks=benchmarks, compute_levels=PILOT_LEVELS,
    n_questions=1, n_samples=1,
)
pilot_df.to_csv("/kaggle/working/h1_pilot_results.csv", index=False)

print("\n" + "=" * 60)
print("[pilot] SANITY CHECKS")
print("=" * 60)

seen = pilot_df.groupby(["benchmark", "difficulty_tier", "task_type"]).size()
missing = {(b.name, b.difficulty_tier, b.task_type) for b in benchmarks} - set(seen.index)
print(f"[pilot] missing benchmarks: {missing if missing else 'none'}")

# Parser sanity: below ~0.9 the thinking/response split is not working and the
# full sweep should not be started.
marker_rate = pilot_df["hit_response_marker"].mean()
print(f"\n[pilot] response-marker hit rate: {marker_rate:.2%} "
      f"({'OK' if marker_rate > 0.9 else 'INVESTIGATE — parsing may still be broken'})")

budget_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["tokens_used"]
    .mean().unstack("compute_level"))
print("\n[pilot] mean completion tokens (total) by compute_level (should rise left->right):")
print(budget_effect)

# thinking_tokens should track compute_level closely; output_tokens should stay
# comparatively flat — a final answer has no reason to grow with the budget.
think_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["thinking_tokens"]
    .mean().unstack("compute_level"))
out_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["output_tokens"]
    .mean().unstack("compute_level"))
print("\n[pilot] mean thinking_tokens by compute_level:")
print(think_effect)
print("\n[pilot] mean output_tokens by compute_level:")
print(out_effect)

bench_acc = pilot_df.groupby(["benchmark", "difficulty_tier"])["is_correct"].mean()
print("\n[pilot] accuracy by benchmark:")
print(bench_acc)

[evaluator] max_model_len=32768 (output capped at 32512) | per-server: KV=[108576, 108576] tokens, max 128 in flight
[resume] 27 calls already in checkpoint — skipping them


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields

[data] GSM8K -> 1 questions
[data] MMLU-Redux -> 1 questions
[data] IFEval -> 1 questions
[data] MATH-full -> 1 questions
[data] MMLU-Pro -> 1 questions
[data] BFCL-v3-simple -> 1 questions
[data] AIME2025 -> 1 questions
[data] GPQA-Diamond -> 1 questions
[data] LiveCodeBench-v6 -> 1 questions
[collect] nothing to do — all calls already done

[pilot] SANITY CHECKS
[pilot] missing benchmarks: none

[pilot] response-marker hit rate: 100.00% (OK)

[pilot] mean completion tokens (total) by compute_level (should rise left->right):
compute_level       0       256     8192
benchmark                               
AIME2025          2000.0  1009.0  4341.0
BFCL-v3-simple      30.0   281.0   156.0
GPQA-Diamond       363.0   332.0  2777.0
GSM8K              217.0   378.0  1614.0
IFEval             234.0   457.0   620.0
LiveCodeBench-v6   159.0   386.0  4131.0
MATH-full          289.0   467.0   788.0
MMLU-Pro            57.0   273.0  2738.0
MMLU-Redux          53.0   266.0   914.0

[pilot] mean thi

## 8. Full sweep

The full grid (§ Research design) for the selected model. Re-running the cell after a dropped session resumes from the checkpoint CSV. When it finishes, the consolidated results are exported as `h1_raw_results.csv` and a summary is printed: rows, tokens, wall time, throughput, overall accuracy, response-marker hit rate, and mean thinking and output tokens.

In [ ]:
# Re-running this cell after a dropped session resumes from the checkpoint CSV.
benchmarks = [
    GSM8KBenchmark(),            # easy / math
    MMLURedux(),                 # easy / general_knowledge
    IFEvalBenchmark(),           # easy / instruction_following
    MATHFullBenchmark(),         # medium / math
    MMLUProBenchmark(),          # medium / general_knowledge
    BFCLv3Benchmark(),           # medium / agentic_tool_use
    AIME2025Benchmark(),         # hard / math
    GPQADiamondBenchmark(),      # hard / phd_science
    LiveCodeBenchV6Benchmark(),  # hard / coding
]

COMPUTE_LEVELS = [0, 256, 512, 1024, 2048, 4096, 8192]
N_QUESTIONS = 100
N_SAMPLES = 5

t_all = time.time()
evaluator = ThinkingBudgetEvaluator(server, max_workers=MAX_WORKERS,
                                    checkpoint="/kaggle/working/h2_raw_results.csv")
raw_df = evaluator.collect(
     benchmarks=benchmarks,
    compute_levels=COMPUTE_LEVELS,
    n_questions=N_QUESTIONS,
    n_samples=N_SAMPLES,
)
# Same empty-string -> NaN round-trip issue as the pilot cell — fix before saving.
raw_df[["raw_model_output", "thinking_text"]] = (
    raw_df[["raw_model_output", "thinking_text"]].fillna(""))

raw_df.to_csv("/kaggle/working/h1_raw_results.csv", index=False)

elapsed_s = time.time() - t_all
total_tok = int(raw_df["tokens_used"].sum())
print("\n" + "=" * 60)
print("[run] FULL SWEEP COMPLETE")
print(f"      rows={len(raw_df)}  tokens={total_tok}")
print(f"      wall={elapsed_s/60:.1f} min  avg={total_tok/max(elapsed_s,1):.0f} tok/s")
print(f"      overall acc={raw_df['is_correct'].mean():.3f}")
# A mid-run drop in the marker hit rate flags e.g. a server restart losing the
# skip_special_tokens override, or prompts pushing budgets past max_model_len.
print(f"      response-marker hit rate={raw_df['hit_response_marker'].mean():.2%}")
print(f"      mean thinking_tokens={raw_df['thinking_tokens'].mean():.0f}  "
      f"mean output_tokens={raw_df['output_tokens'].mean():.0f}")
print(f"      est cost ~${elapsed_s*GPU_RATE_USD_S:.2f} (Kaggle quota covers this)")
print("=" * 60)
raw_df.head()

## 9. Evaluation methodology

All metrics are computed per model, per benchmark and per compute level unless stated otherwise.

| Metric | Definition |
|---|---|
| **Accuracy** | Mean `is_correct` over questions × samples in a (benchmark, budget) cell |
| **Per-question pass rate and stability** | Fraction of the 5 samples correct for a question; questions with rates strictly between 0 and 1 are *unstable* |
| **Knee** | The budget at which the accuracy-versus-log₂(budget) curve flattens, found with the Kneedle algorithm (`kneed`) |
| **Budget utilization** | `thinking_tokens / budget`. Values near 1 mean the cap bound; values well below 1 mean the model stopped on its own |
| **Natural-completion rate** | `1 − mean(was_truncated)` — how often the model finished before the cap |
| **Underthinking** | Rows truncated and incorrect at a budget where the same question is solved at a larger one |
| **Overthinking** | Thinking length keeps growing past the knee with no accuracy gain, especially on questions already solved at small budgets |
| **Tier check** | Low-budget accuracy by benchmark compared against the assumed tier order; mismatches are reported |

The central comparisons are: knee location by tier within a model; utilization and natural-completion curves across budgets; and, between models, accuracy at matched budgets and the position of each model's knee. Cross-model *length* comparisons use words or characters (§3.4).

### Integrity checks and stratified sample

The cells below load the consolidated results, count truncated rows, rows without a closing marker and rows whose marker was found but whose answer is empty, and draw a stratified sample — 10 questions per benchmark × compute level — saved as `sample2.csv` for qualitative reading and for the first pass of the textual analysis (§11).

In [ ]:
raw_df = pd.read_csv("/kaggle/working/h2_raw_results.csv")
raw_df.shape

In [ ]:
print("was_truncated = 1:", (raw_df["was_truncated"] == 1).sum())
print("hit_response_marker = 0:", (raw_df["hit_response_marker"] == 0).sum())
print("hit_marker_empty_output = 1:", (raw_df["hit_marker_empty_output"] == 1).sum())

In [ ]:
N_QUESTIONS_PER_STRATUM = 10

sample_df = (
    raw_df
    .drop_duplicates(
        subset=["benchmark", "compute_level", "question_id"]
    )
    .groupby(["benchmark", "compute_level"], group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(N_QUESTIONS_PER_STRATUM, len(x)),
            random_state=42
        )
    )
    .reset_index(drop=True)
)
sample_df

In [ ]:
sample_df.to_csv("/kaggle/working/sample2.csv", index=False)

## 10. Limitations

- Difficulty tiers are priors; only the low-budget accuracy check turns them into evidence.
- Graders are lenient (§5); absolute scores are not comparable to leaderboards, but curve shapes are.
- `thinking_tokens` and `output_tokens` are approximate and tokenizer-dependent.
- A shared sampling protocol (T = 0.7, no system prompt) differs from each vendor's recommendation.
- Traces cut by the budget are **censored**: their endings reflect the intervention, not the model's choice.
- Five samples per question describe variation within a question; they are not five independent questions.
- The DeepSeek configuration uses fp16 on T4 with a TP = 2 server; a quantized alternative would be a different model.

## 11. What's Next — Textual analysis of `thinking_text`

**Goal.** The sweep answers *how much* each model thinks and *what it buys*. The next stage asks **what the thinking looks like**: which reasoning patterns and characteristics appear in `thinking_text`, how they differ **across the three models** and **across compute budgets**, and which of them separate good thinking from bad. The analysis is descriptive: it discovers properties and associations, not causes.

**Input.** `h1_raw_results.csv` (all models, with the `model` column) and the stratified `sample2.csv` for the first, manually inspectable pass. Run each phase on the sample, validate, then scale to the full dataset.

### Phase 0 — Clean and normalize
- Strip the leading `<think>` (Qwen3) and any boundary residue; normalize whitespace.
- Keep `was_truncated`; add a **`forced_close`** flag for traces where the cap bound, and exclude or down-weight their artificial last sentence.
- Add model-independent size measures: characters, words, sentences, paragraphs.
- Keep the question id so every trace can be paired across budgets, samples and models.

### Phase 1 — Surface profile
- Length distributions by model × budget × correct/incorrect.
- Lexical diversity (type–token ratio), compression ratio (`zlib`) and repeated n-gram rate as cheap proxies for redundancy.
- Script and language mix, share of digits and math symbols, code-like text, and the typical openers and closers of a trace.

### Phase 2 — Discourse-cue profile
- Rates per 1,000 words of reasoning cues (for example *wait, hmm, but, alternatively, let me check, actually, so, therefore*).
- Judge cues **in context**, not as isolated regex hits: "Wait, the answer is …" is a candidate answer, not doubt.
- Compare cue profiles between models and between low and high budgets: does a tighter budget compress the *style* of thinking or only its length?

### Phase 3 — Segment-level structure
- Segment each trace (an LLM segmenter, with paragraph/sentence rules as a fast baseline).
- Assign three independent labels to every segment:
  - **`discourse_role`**: setup · reasoning · verification · doubt · restate · repetition
  - **`answer_status`**: none · candidate · final
  - **`temporal_role`**: new · continuation
- Produce a segment table and a **role-to-role transition matrix** per model, per budget band and per correct/incorrect outcome.

### Phase 4 — Answer trajectory
- Extract candidate answers per segment and locate where the **gold answer first appears** (relative position) and what follows it.
- Trace features: number of answer candidates, answer flips, candidate-to-final, verification after a candidate.
- Trajectory classes: `never_found` · `stable_from_first` · `found_then_lost` · `wavered_but_recovered`.

### Phase 5 — Repetition and looping
- Repeated n-gram loops, segment-to-segment similarity (TF-IDF cosine or embeddings) and a **novelty curve** (new content per segment across the trace).
- Separate useful repetition (re-checking) from harmful repetition (being stuck), and relate both to truncation and correctness.

### Phase 6 — Compare
- Correct vs. incorrect · truncated vs. completed · low vs. high budget · model vs. model on matched question ids and budgets.
- Report effect sizes with confidence intervals, not only means.

### Phase 7 — Validate
- An HTML viewer of representative correct and incorrect traces, showing each segment with its three labels and the extracted answer.
- Hand-check about 30 segments per label and measure agreement between the labeler and manual labels before trusting any aggregate.

### Decisions before starting
- Labeler: a free, local, open model as judge versus a small zero-shot classifier (accuracy versus speed on a very large dataset).
- Segmentation: LLM-based versus rule-based baseline.
- Cross-model length unit: words/characters (recommended) versus a single reference tokenizer.
- Scope of the first pass: all three models on the same question subset, or one model first.
- Sample size per (benchmark, budget, model) cell for the labeled pass.

### Deliverables
`thinking_text_clean.csv` · `segments_labeled.csv` · `trace_features.csv` · transition-matrix and trajectory figures · the HTML validation page · a short written answer to the question: *what do good traces do that bad traces do not, and does that change with the model and the compute budget?*

### Guardrails
- Budget-truncated traces are censored; never read their endings as the model's choice.
- Re-check difficulty tiers with low-budget accuracy before using them as a grouping variable.
- Treat the five samples of a question as correlated, not independent.
- Keep the first pass small and the pipeline simple; scale only after validation.